# ELEC 475 Lab 4: Fine-tuning ResNet50 for CLIP

## For Markers - Running This Notebook

This notebook is designed to run on Kaggle with the following datasets:
1. **MS-COCO 2014 Annotations** (captions): `ms-coco2014`
2. **COCO 2014 Images**: `coco-2014-dataset-for-yolov3`
3. **Pre-encoded Text Embeddings** (generated separately): Upload as your own dataset

### To run this notebook locally or modify paths:

1. **Demo Mode:** Set `DEMO_MODE = True` in the configuration cell to test with only 50 samples (no dataset required)
2. **Local Execution:** The notebook auto-detects if it's running locally and uses `./data/` paths by default
3. **Path Customization:** Modify the path variables in the "Path Configuration" cell below

### Requirements

The COCO 2014 dataset can be downloaded from:
- Captions: https://cocodataset.org/#download (2014 Train/Val annotations)
- Images: https://cocodataset.org/#download (2014 Train/Val images)


## Initial Setup and Path Configuration


In [ ]:
import os
import json
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# --------------------------------------------------------------------
# 1. Environment Detection and Path Configuration
# --------------------------------------------------------------------

# Auto-detect Kaggle environment
IS_KAGGLE = os.path.exists('/kaggle/input')

# DEMO_MODE: Set to True to run with only 50 samples for quick testing
DEMO_MODE = False

if IS_KAGGLE:
    print("✓ Running on Kaggle")
    # Dataset 1: MS-COCO2014 annotations (captions)
    CAPTIONS_ROOT = Path("/kaggle/input/ms-coco2014/annotations")
    # Dataset 2: COCO 2014 images
    IMAGES_ROOT = Path("/kaggle/input/coco-2014-dataset-for-yolov3/coco2014/images")
    # Dataset 3: Pre-encoded text embeddings (modify this to your uploaded dataset name)
    TEXT_EMBEDDINGS_PATH = Path("/kaggle/input/coco-text-embeddings")
    OUTPUT_DIR = Path("/kaggle/working")
else:
    print("✓ Running locally")
    # Modify these paths for local execution
    CAPTIONS_ROOT = Path("./data/annotations")
    IMAGES_ROOT = Path("./data")
    TEXT_EMBEDDINGS_PATH = Path("./data")
    OUTPUT_DIR = Path("./outputs")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set specific paths
CAPTIONS_TRAIN_PATH = CAPTIONS_ROOT / "captions_train2014.json"
CAPTIONS_VAL_PATH = CAPTIONS_ROOT / "captions_val2014.json"
IMAGES_TRAIN_DIR = IMAGES_ROOT / "train2014"
IMAGES_VAL_DIR = IMAGES_ROOT / "val2014"
TRAIN_EMBEDDINGS_PATH = TEXT_EMBEDDINGS_PATH / "train_text_embeddings.pt"
VAL_EMBEDDINGS_PATH = TEXT_EMBEDDINGS_PATH / "val_text_embeddings.pt"

print("Caption files:")
print("  Train:", CAPTIONS_TRAIN_PATH)
print("  Val  :", CAPTIONS_VAL_PATH)
print("\nImage dirs:")
print("  Train images:", IMAGES_TRAIN_DIR)
print("  Val   images:", IMAGES_VAL_DIR)

# --------------------------------------------------------------------
# 2. Helper to load COCO-style caption JSON (robust to a 'root' wrapper)
# --------------------------------------------------------------------
def load_coco_captions(json_path: Path):
    with json_path.open("r") as f:
        data = json.load(f)

    # Some viewers show a "root" wrapper; strip it if present
    if isinstance(data, dict) and "root" in data and isinstance(data["root"], dict):
        data = data["root"]

    # Basic sanity checks
    assert isinstance(data, dict), "Expected top-level JSON object"
    assert "images" in data, "Expected 'images' key in captions file"
    assert "annotations" in data, "Expected 'annotations' key in captions file"

    return data

# --------------------------------------------------------------------
# 3. Load ONLY the caption metadata into RAM (no images)
# --------------------------------------------------------------------
train_captions = load_coco_captions(CAPTIONS_TRAIN_PATH)
val_captions   = load_coco_captions(CAPTIONS_VAL_PATH)

print("\nTrain captions keys:", train_captions.keys())
print("Val captions keys   :", val_captions.keys())

print("\n# train images in captions file     :", len(train_captions["images"]))
print("# train caption annotations entries:", len(train_captions["annotations"]))
print("# val images in captions file      :", len(val_captions["images"]))
print("# val caption annotations entries  :", len(val_captions["annotations"]))

# --------------------------------------------------------------------
# 4. Build image_id -> file_name maps for train / val
# --------------------------------------------------------------------
train_id_to_filename = {img["id"]: img["file_name"] for img in train_captions["images"]}
val_id_to_filename   = {img["id"]: img["file_name"] for img in val_captions["images"]}

# Optional: sanity check file existence for a couple of image IDs
def check_image_exists(image_dir: Path, file_name: str) -> bool:
    return (image_dir / file_name).exists()

some_train_ids = list(train_id_to_filename.keys())[:5]
print("\nSample of train image paths:")
for img_id in some_train_ids:
    fname = train_id_to_filename[img_id]
    path = IMAGES_TRAIN_DIR / fname
    print(f"  id={img_id}, file={fname}, exists={path.exists()}")

# --------------------------------------------------------------------
# 5. Test alignment: show a few random (image, caption) pairs
# --------------------------------------------------------------------
def show_random_captioned_images(
    captions_dict,
    id_to_filename,
    image_dir: Path,
    num_samples: int = 3
):
    ann_list = captions_dict["annotations"]
    print(f"\nShowing {num_samples} random samples from {image_dir.name}...")
    for ann in random.sample(ann_list, num_samples):
        img_id = ann["image_id"]
        caption = ann["caption"]

        fname = id_to_filename.get(img_id, None)
        if fname is None:
            print(f"\n[WARNING] image_id {img_id} not found in 'images' list.")
            continue

        img_path = image_dir / fname

        print("\n--------------------------")
        print(f"image_id: {img_id}")
        print(f"file    : {fname}")
        print(f"path    : {img_path}")
        print(f"caption : {caption}")

        if img_path.exists():
            try:
                img = Image.open(img_path).convert("RGB")
                display(img.resize((256, 256)))
            except Exception as e:
                print(f"Could not open image: {e}")
        else:
            print("[WARNING] Image file does not exist on disk.")

# Check a few training examples
show_random_captioned_images(train_captions, train_id_to_filename, IMAGES_TRAIN_DIR, num_samples=3)

# Check a few validation examples
show_random_captioned_images(val_captions, val_id_to_filename, IMAGES_VAL_DIR, num_samples=3)


## Configuration and Dataset Documentation

### Dataset Configuration

We will use a subset of the COCO 2014 dataset to ensure training completes within Kaggle's time limits.


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Dataset sizes (subset selection)
if DEMO_MODE:
    TRAIN_SUBSET_SIZE = 50
    VAL_SUBSET_SIZE = 20
else:
    TRAIN_SUBSET_SIZE = 200000  # ~50% of full dataset (~414K pairs)
    VAL_SUBSET_SIZE = 40000     # ~20% of full dataset (~200K pairs)

# Training hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 10
TEMPERATURE = 0.07  # Temperature for InfoNCE loss

# Model configuration
RESNET_OUTPUT_DIM = 2048  # ResNet50 final feature dimension
PROJECTION_HIDDEN_DIM = 1024
CLIP_EMBEDDING_DIM = 512  # CLIP embedding space

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Mode: {'DEMO' if DEMO_MODE else 'TRAINING'}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"\nDataset Sizes:")
print(f"  Train subset: {TRAIN_SUBSET_SIZE:,} pairs")
print(f"  Val subset: {VAL_SUBSET_SIZE:,} pairs")
print(f"\nTraining Hyperparameters:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Temperature: {TEMPERATURE}")
print("=" * 60)


### Dataset Information (Lab Documentation Requirements)

**Original Dataset Sizes:**
- Training: ~414K image-caption pairs from ~82K images
- Validation: ~200K image-caption pairs from ~40K images

**Subset Sizes Used:**
- Training: 60,000 pairs (15% of full dataset)
- Validation: 15,000 pairs (7.5% of full dataset)

**Rationale:**
- Full dataset training would take 12+ hours on Kaggle GPU
- Subset ensures completion within Kaggle time limits (9-12 hours)
- Still provides sufficient data for meaningful CLIP fine-tuning
- Not graded on performance, focus is on implementation quality

**Image Preprocessing:**
- Resize to 224×224 pixels
- Normalize with CLIP statistics:
  - Mean: [0.48145466, 0.4578275, 0.40821073]
  - Std: [0.26862954, 0.26130258, 0.27577711]

**Text Preprocessing:**
- Pre-encoded using CLIP text encoder (openai/clip-vit-base-patch32)
- Tokenization: CLIP tokenizer (max length 77, padding/truncation handled)
- Output: 512-dimensional L2-normalized embeddings


## Dataset Class


In [ ]:
# CLIP normalization statistics
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]


def get_clip_image_transform(augment=False):
    """
    Get transform pipeline for images to match CLIP preprocessing.
    
    Args:
        augment: If True, apply data augmentation for training
    """
    if augment:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD)
        ])
    else:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD)
        ])


class COCODataset(Dataset):
    """
    COCO 2014 dataset loader for CLIP fine-tuning.
    
    Loads images and their corresponding pre-encoded caption embeddings.
    Handles alignment issues between caption and image datasets robustly.
    """
    
    def __init__(
        self,
        annotations_path: Path,
        images_dir: Path,
        text_embeddings_path: Path,
        subset_size: int = None,
        transform=None
    ):
        """
        Initialize COCO dataset.
        
        Args:
            annotations_path: Path to COCO captions JSON file
            images_dir: Directory containing images
            text_embeddings_path: Path to pre-encoded text embeddings .pt file
            subset_size: Number of pairs to sample (None for all)
            transform: Image transform pipeline
        """
        self.images_dir = images_dir
        self.transform = transform if transform is not None else get_clip_image_transform()
        
        # Load annotations
        print(f"Loading annotations from {annotations_path}...")
        captions_data = load_coco_captions(annotations_path)
        
        # Build image info dictionary
        images_dict = {img['id']: img for img in captions_data['images']}
        
        # Load text embeddings
        print(f"Loading text embeddings from {text_embeddings_path}...")
        if text_embeddings_path.exists():
            self.text_embeddings_cache = torch.load(text_embeddings_path, map_location='cpu')
            print(f"✓ Loaded embeddings for {len(self.text_embeddings_cache)} images")
        else:
            raise FileNotFoundError(f"Text embeddings not found at {text_embeddings_path}")
        
        # Build valid pairs list (only for images that have embeddings)
        # Note: Skipping file existence checks for speed (takes 10-20 min on Kaggle)
        # Will catch missing images at load time if needed
        print("Building dataset pairs...")
        self.pairs = []
        skipped_no_embedding = 0
        skipped_no_metadata = 0
        caption_counters = {}  # Track caption count per image for O(1) lookup
        
        for ann in captions_data['annotations']:
            image_id = ann['image_id']
            
            # Check if we have embeddings for this image (fast - in memory)
            if image_id not in self.text_embeddings_cache:
                skipped_no_embedding += 1
                continue
                
            # Check if image metadata exists (fast - dictionary lookup)
            if image_id not in images_dict:
                skipped_no_metadata += 1
                continue
                
            image_info = images_dict[image_id]
            
            # Find caption index - O(1) instead of O(n)
            caption_idx = caption_counters.get(image_id, 0)
            caption_counters[image_id] = caption_idx + 1
            
            self.pairs.append((image_id, caption_idx, image_info['file_name']))
        
        print(f"✓ Built dataset with {len(self.pairs)} pairs")
        if skipped_no_embedding > 0:
            print(f"  Skipped {skipped_no_embedding} annotations (no embedding)")
        if skipped_no_metadata > 0:
            print(f"  Skipped {skipped_no_metadata} annotations (no metadata)")
        
        # Sample subset if requested
        if subset_size is not None and subset_size < len(self.pairs):
            random.seed(42)  # For reproducibility
            self.pairs = random.sample(self.pairs, subset_size)
            print(f"✓ Sampled {subset_size} pairs from dataset")
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        """
        Get a sample from the dataset.
        
        Returns:
            tuple: (image_tensor, caption_embedding)
                - image_tensor: [3, 224, 224] normalized with CLIP stats
                - caption_embedding: [512] pre-encoded CLIP text embedding
        """
        image_id, caption_idx, file_name = self.pairs[idx]
        
        # Load and transform image
        image_path = self.images_dir / file_name
        try:
            image = Image.open(image_path).convert('RGB')
            if self.transform is not None:
                image = self.transform(image)
        except (FileNotFoundError, OSError) as e:
            # If image is missing, skip to next sample
            # This is rare but handles edge cases from skipped existence checks
            return self.__getitem__((idx + 1) % len(self.pairs))
        
        # Get caption embedding
        caption_embedding = self.text_embeddings_cache[image_id][caption_idx]
        
        return image, caption_embedding


In [ ]:
print("=" * 60)
print("LOADING DATASETS")
print("=" * 60)

# Create training dataset
train_dataset = COCODataset(
    annotations_path=CAPTIONS_TRAIN_PATH,
    images_dir=IMAGES_TRAIN_DIR,
    text_embeddings_path=TRAIN_EMBEDDINGS_PATH,
    subset_size=TRAIN_SUBSET_SIZE,
    transform=get_clip_image_transform(augment=True)  # ← Add this parameter
)

# Create validation dataset (no augmentation for val)
val_dataset = COCODataset(
    annotations_path=CAPTIONS_VAL_PATH,
    images_dir=IMAGES_VAL_DIR,
    text_embeddings_path=VAL_EMBEDDINGS_PATH,
    subset_size=VAL_SUBSET_SIZE,
    transform=get_clip_image_transform(augment=False)  # ← Add this parameter
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\n✓ Training set: {len(train_dataset)} pairs, {len(train_loader)} batches")
print(f"✓ Validation set: {len(val_dataset)} pairs, {len(val_loader)} batches")
print("=" * 60)


## Model Architecture

We implement a ResNet50 image encoder with a projection head to map images into CLIP's 512-dimensional embedding space.


In [ ]:
class CLIPImageEncoder(nn.Module):
    """
    Image encoder for CLIP fine-tuning.
    
    Uses ResNet50 (pretrained on ImageNet) as backbone with a projection head
    to map image features to CLIP's 512-dimensional embedding space.
    """
    
    def __init__(
        self,
        resnet_output_dim: int = 2048,
        projection_hidden_dim: int = 1024,
        clip_embedding_dim: int = 512
    ):
        """
        Initialize the image encoder.
        
        Args:
            resnet_output_dim: Output dimension of ResNet50 (2048 for ResNet50)
            projection_hidden_dim: Hidden dimension of projection head
            clip_embedding_dim: CLIP embedding dimension (512)
        """
        super().__init__()
        
        # Load pretrained ResNet50
        resnet = torchvision.models.resnet50(pretrained=True)
        
        # Remove the final classification layer
        # ResNet50 without fc layer outputs 2048-dim features
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        # Projection head: 2048 -> 1024 -> 512
        # Using GELU activation as per lab requirements
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(resnet_output_dim, projection_hidden_dim),
            nn.GELU(),
            nn.Linear(projection_hidden_dim, clip_embedding_dim)
        )
        
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Input images [batch_size, 3, 224, 224]
            
        Returns:
            Image embeddings [batch_size, 512], L2-normalized
        """
        # Extract features with ResNet50
        features = self.backbone(x)  # [batch_size, 2048, 1, 1]
        
        # Project to CLIP embedding space
        embeddings = self.projection(features)  # [batch_size, 512]
        
        # L2 normalize (CLIP uses normalized embeddings)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        
        return embeddings


# Initialize model
model = CLIPImageEncoder(
    resnet_output_dim=RESNET_OUTPUT_DIM,
    projection_hidden_dim=PROJECTION_HIDDEN_DIM,
    clip_embedding_dim=CLIP_EMBEDDING_DIM
).to(DEVICE)

# Enable multi-GPU training if available
num_gpus = torch.cuda.device_count()
if num_gpus > 1:
    print(f"\n✓ Found {num_gpus} GPUs! Using DataParallel for multi-GPU training")
    model = nn.DataParallel(model)
    print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(num_gpus)]}")
elif num_gpus == 1:
    print(f"\n✓ Using single GPU: {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠ No GPU detected, using CPU")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "=" * 60)
print("MODEL ARCHITECTURE")
print("=" * 60)
print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("=" * 60)


## InfoNCE Loss

The InfoNCE (contrastive) loss maximizes agreement between positive (aligned) image-text pairs while minimizing agreement with negative (misaligned) pairs.


In [ ]:
def info_nce_loss(image_embeddings, text_embeddings, temperature=0.07):
    """
    Compute InfoNCE (contrastive) loss for CLIP.
    
    The loss encourages the model to maximize similarity between paired
    image-text embeddings while minimizing similarity with other pairs in the batch.
    
    Args:
        image_embeddings: [batch_size, embedding_dim] - L2 normalized image embeddings
        text_embeddings: [batch_size, embedding_dim] - L2 normalized text embeddings
        temperature: Temperature scaling parameter (default: 0.07)
        
    Returns:
        loss: Scalar loss value
    """
    batch_size = image_embeddings.shape[0]
    
    # Compute similarity matrix: [batch_size, batch_size]
    # Higher values = more similar
    logits = torch.matmul(image_embeddings, text_embeddings.T) / temperature
    
    # Labels: diagonal elements are positive pairs
    labels = torch.arange(batch_size, device=image_embeddings.device)
    
    # Symmetric loss: image-to-text + text-to-image
    loss_i2t = F.cross_entropy(logits, labels)  # Image to text
    loss_t2i = F.cross_entropy(logits.T, labels)  # Text to image
    
    loss = (loss_i2t + loss_t2i) / 2
    
    return loss


# Test the loss function with dummy data
print("Testing InfoNCE loss function...")
dummy_img_emb = torch.randn(4, 512).to(DEVICE)
dummy_txt_emb = torch.randn(4, 512).to(DEVICE)
dummy_img_emb = F.normalize(dummy_img_emb, p=2, dim=1)
dummy_txt_emb = F.normalize(dummy_txt_emb, p=2, dim=1)
test_loss = info_nce_loss(dummy_img_emb, dummy_txt_emb, temperature=TEMPERATURE)
print(f"✓ InfoNCE loss test: {test_loss.item():.4f}")


## Training Setup


In [ ]:
# Optimizer: AdamW with weight decay
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler: Cosine annealing
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'lr': []
}

print("=" * 60)
print("TRAINING SETUP")
print("=" * 60)
print(f"Optimizer: AdamW")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"Scheduler: CosineAnnealingLR")
print(f"  T_max: {NUM_EPOCHS}")
print(f"  eta_min: 1e-6")
print("=" * 60)


## Training Loop


In [ ]:
def train_epoch(model, loader, optimizer, temperature, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    num_batches = 0
    
    pbar = tqdm(loader, desc="Training")
    for images, text_embeddings in pbar:
        images = images.to(device)
        text_embeddings = text_embeddings.to(device)
        
        # Forward pass
        image_embeddings = model(images)
        loss = info_nce_loss(image_embeddings, text_embeddings, temperature)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        num_batches += 1
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / num_batches


def validate(model, loader, temperature, device):
    """Validate the model."""
    model.eval()
    total_loss = 0
    num_batches = 0
    
    with torch.no_grad():
        pbar = tqdm(loader, desc="Validation")
        for images, text_embeddings in pbar:
            images = images.to(device)
            text_embeddings = text_embeddings.to(device)
            
            # Forward pass
            image_embeddings = model(images)
            loss = info_nce_loss(image_embeddings, text_embeddings, temperature)
            
            total_loss += loss.item()
            num_batches += 1
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / num_batches


# Training loop
print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

start_time = time.time()
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 60)
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, TEMPERATURE, DEVICE)
    
    # Validate
    val_loss = validate(model, val_loader, TEMPERATURE, DEVICE)
    
    # Update learning rate
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(current_lr)
    
    # Print epoch summary
    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Learning Rate: {current_lr:.6f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint_path = OUTPUT_DIR / 'best_model.pt'
        # Handle DataParallel wrapper when saving
        model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model_state,
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, checkpoint_path)
        print(f"  ✓ Saved best model (val_loss: {val_loss:.4f})")

training_time = time.time() - start_time

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Total training time: {training_time / 60:.2f} minutes")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Final training loss: {train_loss:.4f}")
print(f"Final validation loss: {val_loss:.4f}")
print("=" * 60)


## Results and Visualization


In [ ]:
# Plot training and validation loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(range(1, NUM_EPOCHS + 1), history['train_loss'], 'b-o', label='Training Loss', linewidth=2)
axes[0].plot(range(1, NUM_EPOCHS + 1), history['val_loss'], 'r-o', label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Learning rate curve
axes[1].plot(range(1, NUM_EPOCHS + 1), history['lr'], 'g-o', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved training curves to {OUTPUT_DIR / 'training_curves.png'}")


In [ ]:
print("=" * 60)
print("HARDWARE AND TRAINING INFORMATION")
print("=" * 60)

# Hardware information
print("\nHardware:")
print(f"  Device: {DEVICE}")
gpu_name = "N/A"
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"  GPU: {gpu_name}")
    print(f"  CUDA Version: {torch.version.cuda}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("  Running on CPU")

# Training statistics
print("\nTraining Statistics:")
print(f"  Total epochs: {NUM_EPOCHS}")
print(f"  Total training time: {training_time / 60:.2f} minutes ({training_time / 3600:.2f} hours)")
print(f"  Average time per epoch: {training_time / NUM_EPOCHS / 60:.2f} minutes")
print(f"  Training samples: {len(train_dataset):,}")
print(f"  Validation samples: {len(val_dataset):,}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Total training iterations: {NUM_EPOCHS * len(train_loader):,}")

# Final results
print("\nFinal Results:")
best_epoch = history['val_loss'].index(best_val_loss) + 1
print(f"  Best validation loss: {best_val_loss:.4f} (epoch {best_epoch})")
print(f"  Final training loss: {history['train_loss'][-1]:.4f}")
print(f"  Final validation loss: {history['val_loss'][-1]:.4f}")

# Model information
print("\nModel Information:")
print(f"  Architecture: ResNet50 + Projection Head")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Image input size: 224×224")
print(f"  Embedding dimension: {CLIP_EMBEDDING_DIM}")

print("=" * 60)

# Save all metrics to a text file for easy reference
summary_path = OUTPUT_DIR / 'training_summary.txt'
with open(summary_path, 'w') as f:
    f.write("=" * 60 + "\n")
    f.write("CLIP FINE-TUNING TRAINING SUMMARY\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("HARDWARE INFORMATION\n")
    f.write("-" * 60 + "\n")
    f.write(f"Device: {DEVICE}\n")
    f.write(f"GPU: {gpu_name}\n")
    if torch.cuda.is_available():
        f.write(f"CUDA Version: {torch.version.cuda}\n")
        f.write(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB\n")
    f.write("\n")
    
    f.write("DATASET INFORMATION\n")
    f.write("-" * 60 + "\n")
    f.write(f"Training samples: {len(train_dataset):,}\n")
    f.write(f"Validation samples: {len(val_dataset):,}\n")
    f.write(f"Batch size: {BATCH_SIZE}\n")
    f.write(f"Total batches per epoch: {len(train_loader)}\n")
    f.write("\n")
    
    f.write("MODEL ARCHITECTURE\n")
    f.write("-" * 60 + "\n")
    f.write(f"Architecture: ResNet50 (ImageNet pretrained) + Projection Head\n")
    f.write(f"Projection: 2048 -> {PROJECTION_HIDDEN_DIM} -> {CLIP_EMBEDDING_DIM}\n")
    f.write(f"Activation: GELU\n")
    f.write(f"Total parameters: {total_params:,}\n")
    f.write(f"Trainable parameters: {trainable_params:,}\n")
    f.write(f"Image input size: 224×224\n")
    f.write(f"Embedding dimension: {CLIP_EMBEDDING_DIM}\n")
    f.write("\n")
    
    f.write("TRAINING CONFIGURATION\n")
    f.write("-" * 60 + "\n")
    f.write(f"Optimizer: AdamW\n")
    f.write(f"Learning rate: {LEARNING_RATE}\n")
    f.write(f"Weight decay: {WEIGHT_DECAY}\n")
    f.write(f"Temperature: {TEMPERATURE}\n")
    f.write(f"Total epochs: {NUM_EPOCHS}\n")
    f.write(f"LR Scheduler: CosineAnnealingLR\n")
    f.write("\n")
    
    f.write("TRAINING RESULTS\n")
    f.write("-" * 60 + "\n")
    f.write(f"Total training time: {training_time / 60:.2f} minutes ({training_time / 3600:.2f} hours)\n")
    f.write(f"Average time per epoch: {training_time / NUM_EPOCHS / 60:.2f} minutes\n")
    f.write(f"Total training iterations: {NUM_EPOCHS * len(train_loader):,}\n")
    f.write("\n")
    f.write(f"Best validation loss: {best_val_loss:.4f} (epoch {best_epoch})\n")
    f.write(f"Final training loss: {history['train_loss'][-1]:.4f}\n")
    f.write(f"Final validation loss: {history['val_loss'][-1]:.4f}\n")
    f.write("\n")
    
    f.write("LOSS HISTORY (PER EPOCH)\n")
    f.write("-" * 60 + "\n")
    f.write("Epoch | Train Loss | Val Loss   | LR\n")
    f.write("-" * 60 + "\n")
    for i in range(NUM_EPOCHS):
        f.write(f"{i+1:5d} | {history['train_loss'][i]:10.4f} | {history['val_loss'][i]:10.4f} | {history['lr'][i]:.2e}\n")
    f.write("\n")
    
    f.write("=" * 60 + "\n")

print(f"\n✓ Saved training summary to {summary_path}")


## Sample Predictions


In [ ]:
def denormalize_image(tensor, mean=CLIP_MEAN, std=CLIP_STD):
    """Denormalize a normalized image tensor for visualization."""
    tensor = tensor.clone()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    tensor = torch.clamp(tensor, 0, 1)
    return tensor.permute(1, 2, 0).cpu().numpy()


def show_sample_predictions(model, dataset, num_samples=6):
    """Display sample images with their similarity scores to captions."""
    model.eval()
    
    # Randomly select samples
    indices = random.sample(range(len(dataset)), num_samples)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    with torch.no_grad():
        for idx, ax in zip(indices, axes):
            image, text_embedding = dataset[idx]
            image_id, caption_idx, file_name = dataset.pairs[idx]
            
            # Get image embedding from model
            image_batch = image.unsqueeze(0).to(DEVICE)
            image_embedding = model(image_batch)
            
            # Compute similarity (cosine similarity since embeddings are normalized)
            text_embedding = text_embedding.unsqueeze(0).to(DEVICE)
            similarity = torch.matmul(image_embedding, text_embedding.T).item()
            
            # Get caption text from original annotations
            # (We need to reload this since we only have embeddings)
            # For visualization purposes, we'll just show the similarity score
            
            # Denormalize and display image
            img_display = denormalize_image(image)
            ax.imshow(img_display)
            ax.axis('off')
            ax.set_title(f'Image ID: {image_id}\\nSimilarity: {similarity:.3f}', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Saved sample predictions to {OUTPUT_DIR / 'sample_predictions.png'}")


# Show samples from validation set
print("Generating sample predictions from validation set...")
show_sample_predictions(model, val_dataset, num_samples=6)


## Issues and Observations

### Observed Issues During Training

**Common issues that may occur:**

1. **Loss Divergence/Instability:**
   - If training loss increases or becomes unstable, reduce learning rate
   - Consider using gradient clipping: `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`
   - Reduce batch size if memory issues occur

2. **Overfitting:**
   - If validation loss increases while training loss decreases, the model is overfitting
   - Solutions: Early stopping, data augmentation, reduce model capacity

3. **Slow Convergence:**
   - If loss decreases very slowly, try increasing learning rate
   - Ensure data is properly shuffled
   - Check that embeddings are properly normalized

4. **Memory Issues:**
   - Reduce batch size
   - Reduce number of data loader workers
   - Use gradient accumulation if needed

### How Issues Were Addressed (Report This)

Document any issues you encountered during your training run and how you resolved them:
- Example: "Initial learning rate of 1e-3 caused loss divergence, reduced to 1e-4"
- Example: "Validation loss started increasing after epoch 5, used early stopping"
- Example: "GPU memory limit reached with batch size 128, reduced to 64"


## Summary and Deliverables

All required outputs have been saved to the output directory for download/submission.


In [ ]:
print("=" * 60)
print("DELIVERABLES SUMMARY")
print("=" * 60)

print(f"\nOutput directory: {OUTPUT_DIR}")
print("\nFiles saved:")
print("  1. best_model.pt - Best model checkpoint")
print("  2. training_summary.txt - Complete training metrics and timing")
print("  3. training_curves.png - Loss and learning rate curves")
print("  4. sample_predictions.png - Sample predictions visualization")

print("\nFor your lab report, include:")
print("  ✓ Training and validation loss curves (saved)")
print("  ✓ Total training time and hardware used (printed above)")
print("  ✓ Dataset information (documented in markdown cells)")
print("  ✓ Model architecture details (ResNet50 + projection head)")
print("  ✓ Preprocessing details (image: 224×224 + CLIP normalization)")
print("  ✓ Text preprocessing (CLIP tokenizer, 512-dim embeddings)")
print("  ✓ Any issues encountered and solutions (document in report)")

print("\n" + "=" * 60)
print("LAB 4 COMPLETE!")
print("=" * 60)
